# 03.7 Common Beginner Syntax Errors

Error messages are not punishment — they are the most precise feedback you will
ever get. Learning to read them turns a frustrating hunt into a ten-second fix.

This notebook triggers each common error deliberately, so you can read the real
message rather than a description of it.

## Theory

### Two categories, two very different timings

This distinction comes straight from Chapter 01, and it explains *when* you find
out about a mistake:

- **Syntax errors** are found at **compile time**, before any code runs. Your
  whole file fails. Nothing executes — not even line 1.
- **Runtime errors** are found while the program is running. Everything up to
  that line already happened.

So if you see *some* output before the error, it is a runtime error. If you see
none at all, it is a syntax error.

### How to read a traceback

Python prints the call stack from **outermost to innermost**, so:

> **Read a traceback from the bottom up.**

The last line is the error type and message. The line above it is the code that
failed. Everything higher is how you got there.

### The caret line

Python 3.10+ underlines the exact span of the problem with `^` characters. Where
it points is the position where Python *realised* something was wrong — which is
often just **after** the real mistake. An unclosed bracket on line 4 is typically
reported on line 5.

### The most useful habit

When an error names a line that looks correct, **check the line above it.**
Unclosed brackets, missing commas, and missing colons all report on the following
line.

In [ ]:
# A helper we reuse throughout this notebook.
def show_error(label, source, note=""):
    """Compile a broken snippet and print the error Python gives."""
    print("CASE:", label)

    # Show the offending source with line numbers.
    for line_number, line in enumerate(source.rstrip(chr(10)).split(chr(10)), start=1):
        print(f"   {line_number} | {line}")

    try:
        compile(source, "<demo>", "exec")
        print("   -> compiled without error")
    except SyntaxError as error:
        # error.msg is the short reason; error.lineno is where Python noticed.
        print(f"   -> {type(error).__name__}: {error.msg}")
        print(f"      reported on line {error.lineno}")

    if note:
        print("      NOTE:", note)
    print("")


NEWLINE = chr(10)
print("Helper ready.")

## The seven most common syntax errors

Each of these is triggered for real below, so the message you see is the message
you will get.

In [ ]:
# 1. Missing colon after a block opener.
show_error(
    "missing colon",
    "if True" + NEWLINE + "    print('hi')" + NEWLINE,
    "every block opener ends with a colon",
)

# 2. Using = instead of == in a condition.
show_error(
    "assignment in a condition",
    "x = 5" + NEWLINE + "if x = 5:" + NEWLINE + "    print('hi')" + NEWLINE,
    "= assigns, == compares",
)

# 3. Unclosed bracket - note which line is reported.
show_error(
    "unclosed bracket",
    "values = [1, 2, 3" + NEWLINE + "print('after')" + NEWLINE,
    "reported on line 2, but the mistake is on line 1",
)

# 4. Unterminated string.
show_error(
    "unterminated string",
    "message = 'hello" + NEWLINE,
    "the quote was never closed",
)

In [ ]:
# 5. Missing comma between collection items.
show_error(
    "missing comma",
    "values = [1 2, 3]" + NEWLINE,
    "Python suggests the fix in the message itself",
)

# 6. Python 2 print syntax.
show_error(
    "Python 2 print",
    "print 'hello'" + NEWLINE,
    "print is a function in Python 3 - it needs brackets",
)

# 7. Invalid identifier.
show_error(
    "name starting with a digit",
    "2items = 10" + NEWLINE,
    "identifiers cannot start with a digit - see 03.5",
)

### Notice case 3

The unclosed bracket was **reported on line 2**, but the mistake is on line 1.
Python kept reading, expecting the bracket to close, and only gave up when it hit
something that could not possibly continue the expression.

This is the single most useful debugging habit from this notebook: **when the
reported line looks fine, look at the line above it.**

## Runtime errors: the ones that happen while running

These compile perfectly. They fail only when execution reaches them — which is
why you see output before the error.

In [ ]:
def show_runtime_error(label, action, note=""):
    """Run something that fails and report the error."""
    print("CASE:", label)
    try:
        action()
        print("   -> no error")
    except Exception as error:
        print(f"   -> {type(error).__name__}: {error}")
    if note:
        print("      NOTE:", note)
    print("")


# NameError - using a name that does not exist.
show_runtime_error(
    "NameError",
    lambda: undefined_variable_name,
    "usually a typo, or a name defined in another cell",
)

# TypeError - an operation on the wrong type.
show_runtime_error(
    "TypeError (str + int)",
    lambda: "5" + 5,
    "the classic input() bug - convert with int() first",
)

# ValueError - right type, impossible value.
show_runtime_error(
    "ValueError",
    lambda: int("not a number"),
    "the type is right (str) but the content is not convertible",
)

# ZeroDivisionError
show_runtime_error(
    "ZeroDivisionError",
    lambda: 10 / 0,
    "check the divisor before dividing",
)

In [ ]:
# IndexError - position past the end of a sequence.
show_runtime_error(
    "IndexError",
    lambda: [1, 2, 3][10],
    "valid indices for a 3-item list are 0, 1, 2 (and -1, -2, -3)",
)

# KeyError - a dictionary key that is not present.
show_runtime_error(
    "KeyError",
    lambda: {"a": 1}["missing"],
    "use .get() when a key may legitimately be absent",
)

# AttributeError - the object has no such attribute or method.
show_runtime_error(
    "AttributeError",
    lambda: "text".push("x"),
    "strings have .join and .split, not .push - check with dir()",
)

# ModuleNotFoundError - the import target is not installed.
def import_missing():
    import definitely_not_installed_xyz

show_runtime_error(
    "ModuleNotFoundError",
    import_missing,
    "install it, or check you are in the right virtual environment",
)

## Reading a real traceback

A traceback with several frames shows the chain of calls. Read it from the
bottom.

In [ ]:
import traceback


def load_settings(raw):
    """Convert a raw value into a number of retries."""
    # This is where the failure actually happens.
    return int(raw)


def configure(config):
    """Pull the retry setting out of a config dictionary."""
    return load_settings(config["retries"])


def start_app():
    """Entry point."""
    return configure({"retries": "three"})


try:
    start_app()
except ValueError:
    # format_exc() gives the traceback as a string so we can annotate it.
    lines = traceback.format_exc().strip().split(chr(10))

    print("THE TRACEBACK:")
    for line in lines:
        print("   " + line)

    print("")
    print("HOW TO READ IT - bottom up:")
    print("   1. Last line: the error type and message")
    print("      -> ValueError: invalid literal for int() with base 10: 'three'")
    print("   2. Line above: the code that failed")
    print("      -> return int(raw), inside load_settings")
    print("   3. Above that: who called it")
    print("      -> configure called load_settings")
    print("      -> start_app called configure")
    print("")
    print("   The FIX belongs wherever the bad value entered - here, the")
    print("   config dictionary containing 'three' instead of 3.")

## Errors that are hard to spot

These produce messages that do not obviously point at the real problem.

In [ ]:
# 1. A missing comma silently concatenates two strings (from 03.4).
names = ["alpha", "beta" "gamma"]
print("1. Missing comma in a list of strings:")
print("   ", names, "->", len(names), "items, not 3. No error raised.")

# 2. Mutable default argument - the Chapter 16 trap, previewed here.
def add_item(item, target=[]):
    """Append to a list that defaults to empty."""
    target.append(item)
    return target

print("")
print("2. Mutable default argument:")
print("   first call: ", add_item("a"))
print("   second call:", add_item("b"), "<- 'a' is still there")
print("   The default list was created ONCE, at definition time.")

# 3. Integer division when you wanted true division.
total = 7
count = 2
print("")
print("3. Wrong division operator:")
print("   7 // 2 =", total // count, "  (floor - often not what you want)")
print("   7 / 2  =", total / count, " (true division)")

# 4. Comparing floats for exact equality.
print("")
print("4. Float equality:")
print("   0.1 + 0.2 == 0.3 ->", 0.1 + 0.2 == 0.3)
print("   actual sum:", 0.1 + 0.2)
print("   Use math.isclose() instead. Chapter 05 explains why.")

## A diagnostic table

When you have an error message and no idea where to start, work down this list.

In [ ]:
diagnostics = [
    ("SyntaxError: invalid syntax", "check the line ABOVE for an unclosed bracket"),
    ("SyntaxError: expected ':'", "a block opener is missing its colon"),
    ("IndentationError", "mixed tabs and spaces, or an inconsistent level"),
    ("NameError", "typo, or used before assignment, or another cell not run"),
    ("TypeError", "wrong type - print type(x) to see what you actually have"),
    ("ValueError", "right type, impossible value - check the input data"),
    ("AttributeError", "wrong type, or a typo - run dir(x) to see what exists"),
    ("KeyError", "key absent - use .get() or check with `in` first"),
    ("IndexError", "off by one - remember indexing starts at 0"),
    ("ModuleNotFoundError", "not installed, or the wrong virtual environment"),
    ("ZeroDivisionError", "guard the divisor before dividing"),
    ("RecursionError", "a base case is missing or never reached"),
]

print("Error                              First thing to check")
print("-" * 92)
for error, check in diagnostics:
    print(error.ljust(35), check)

## The debugging procedure

When you are stuck, this order resolves most problems quickly.

In [ ]:
steps = [
    "Read the LAST line of the traceback - the type and message",
    "Read the line above it - the code that failed",
    "If that line looks correct, check the line BEFORE it",
    "Print the types: print(type(x), type(y))",
    "Print the values: print(repr(x)) - repr shows quotes and whitespace",
    "Check your assumptions: is the list empty? is the key present?",
    "Reduce it: cut the code down until the error disappears",
    "Search the exact message text, without your variable names",
]

print("When stuck, in order:")
print("-" * 66)
for index, step in enumerate(steps, start=1):
    print(f"  {index}. {step}")

print("")
print("Step 5 matters more than it looks. repr() reveals what print() hides:")

value = "42 "
print("   print(value)       ->", value, "<- looks like a clean number")
print("   print(repr(value)) ->", repr(value), "<- a trailing space, and a string")

print("")
print("That trailing space is why int(value) would still work but a")
print("comparison against '42' would fail. repr() shows it immediately.")

## Takeaways

1. **Syntax errors** stop the file compiling — you see *no* output. **Runtime
   errors** happen mid-execution — you see output first.
2. Read tracebacks **from the bottom up**: type, then failing line, then callers.
3. When the reported line looks correct, **check the line above** — unclosed
   brackets and missing commas report on the following line.
4. The caret `^` marks where Python *noticed*, not always where you erred.
5. `print(type(x))` and `print(repr(x))` resolve most confusing errors in
   seconds.
6. Some mistakes raise **no error at all** — a missing comma between strings, a
   mutable default argument, `//` instead of `/`.
7. Error messages are precise. Read them fully before changing anything.

## Try it yourself

1. Trigger each of the seven syntax errors yourself and read the real messages.
2. Write a three-function call chain that fails in the innermost one. Read the
   traceback bottom-up and identify where the bad value entered.
3. Take a string with a trailing space and compare `print()` with `print(repr())`.
4. Deliberately omit a comma between two strings in a list. How long does it take
   you to spot it without a linter?